# SGLD based light YOLO

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# Model and loss

In [4]:
def conv_block(x, filters, kernel=3, stride=1):
    x = layers.Conv2D(filters, kernel, stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)
    return x

In [10]:
def tiny_yolo_like(input_shape=(128, 128, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    x = conv_block(inputs, 16)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 32)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 64)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 128)
    x = layers.MaxPooling2D(2)(x)
    x = conv_block(x, 256)

    x = layers.Conv2D(num_classes * 5, 1, padding='same')(x)
    outputs = layers.Activation('sigmoid')(x)
    return Model(inputs, outputs)

In [12]:
model = tiny_yolo_like()

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 128, 128, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 128, 128, 16)        │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 128, 128, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu (LeakyReLU)              │ (None, 128, 128, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 64, 64, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 64, 64, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 64, 64, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_1 (LeakyReLU)            │ (None, 64, 64, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 32, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 32, 32, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 32, 32, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_2 (LeakyReLU)            │ (None, 32, 32, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 16, 16, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 16, 16, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 16, 16, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_3 (LeakyReLU)            │ (None, 16, 16, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 8, 8, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 8, 8, 256)           │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 8, 8, 256)           │           1,024 │
│ (BatchNormalization)                 │                             │              

 Total params: 395,877 (1.51 MB)

 Trainable params: 394,885 (1.51 MB)

 Non-trainable params: 992 (3.88 KB)

In [16]:
def yolo_loss(y_true, y_pred):
    coord_loss = tf.reduce_mean(tf.square(y_true[..., :4] - y_pred[..., :4]))
    obj_loss = tf.reduce_mean(tf.keras.losses.binary_crossentropy(
        y_true[..., 4:], y_pred[..., 4:]
    ))
    return coord_loss + obj_loss

# SGLD

In [20]:
lr = 1e-5
num_steps = 2000
num_samples = num_steps // 10

In [24]:
trainable_vars = model.trainable_variables

In [28]:
samples_ta = tf.TensorArray(dtype=tf.float32, size=num_samples)
sample_idx = 0

In [30]:
@tf.function
def sgld_step(x, y):
    with tf.GradientTape() as tape:
        pred = model(x, training=True)
        loss = yolo_loss(y, pred)
    grads = tape.gradient(loss, trainable_vars)
    for var, grad in zip(trainable_vars, grads):
        noise = tf.random.normal(shape=var.shape, stddev=tf.sqrt(lr))
        var.assign_sub(0.5 * lr * grad - noise)
    return loss

In [32]:
for step in range(num_steps):
    x_batch = tf.random.normal([4, 128, 128, 3])
    y_batch = tf.random.uniform([4, 8, 8, 5], 0, 1)

    loss = sgld_step(x_batch, y_batch)
    if step % 10 == 0:
        print(f"step {step}, loss {loss.numpy():.4f}")
        flat = tf.concat([tf.reshape(v, [-1]) for v in trainable_vars], axis=0)
        samples_ta = samples_ta.write(sample_idx, flat)
        sample_idx += 1
samples = samples_ta.stack()

step 0, loss 0.9510
step 10, loss 0.9422
step 20, loss 0.9152
step 30, loss 0.9453
step 40, loss 0.9567
step 50, loss 0.9536
step 60, loss 0.9603
step 70, loss 0.9637
step 80, loss 0.9578
step 90, loss 0.9360
step 100, loss 0.9316
step 110, loss 0.9653
step 120, loss 0.9836
step 130, loss 0.9543
step 140, loss 0.9547
step 150, loss 0.9343
step 160, loss 0.9247
step 170, loss 0.9262
step 180, loss 0.9505
step 190, loss 0.9212
step 200, loss 0.9479
step 210, loss 0.9520
step 220, loss 0.9501
step 230, loss 0.9953
step 240, loss 0.9865
step 250, loss 0.9774
step 260, loss 1.0217
step 270, loss 0.9758
step 280, loss 1.0281
step 290, loss 0.9656
step 300, loss 1.0102
step 310, loss 0.9998
step 320, loss 1.0236
step 330, loss 1.0011
step 340, loss 1.0449
step 350, loss 1.0342
step 360, loss 0.9943
step 370, loss 1.0166
step 380, loss 1.0216
step 390, loss 1.0399
step 400, loss 1.0620
step 410, loss 1.0756
step 420, loss 1.0738
step 430, loss 1.0768
step 440, loss 1.0503
step 450, loss 1.0041

In [34]:
def predict_bayesian(x, samples, model):
    preds = []
    offset = 0
    flat_vars = tf.concat([tf.reshape(v, [-1]) for v in model.trainable_variables], axis=0)
    shapes = [v.shape for v in model.trainable_variables]
    sizes = [tf.size(v) for v in model.trainable_variables]

    for s in samples:
        idx = 0
        for var, shape, size in zip(model.trainable_variables, shapes, sizes):
            new_val = tf.reshape(s[idx:idx+size], shape)
            var.assign(new_val)
            idx += size
        preds.append(model(x, training=False))
    return tf.reduce_mean(preds, axis=0), tf.math.reduce_std(preds, axis=0)

In [36]:
x_test = tf.random.normal([1, 128, 128, 3])
mean_pred, std_pred = predict_bayesian(x_test, samples[-50:], model)

In [38]:
print(mean_pred.shape, std_pred.shape)

(1, 8, 8, 5) (1, 8, 8, 5)
